In [1]:
##### interface to test the model on unseen data, change your trained model file path and upload recorded audio .wav "unseen" 
import tkinter as tk
from tkinter import filedialog, messagebox
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import torch
import librosa
import joblib
import numpy as np

# ==========================
# Load Wav2Vec2 and your trained classifier
# ==========================
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")

# Load your pre-trained model from Downloads
# clf = joblib.load(r"C:\Users\yasmi\Downloads\speech_issue_model.joblib")
clf = joblib.load(r"/Users/linabenna/Desktop/ai4g/speech_issue_model.joblib")

# ==========================
# Function to predict audio
# ==========================
def predict_audio(file_path):
    y_audio, sr = librosa.load(file_path, sr=16000)
    inputs = processor(y_audio, sampling_rate=sr, return_tensors="pt")
    with torch.no_grad():
        emb = model(**inputs).last_hidden_state.mean(dim=1).numpy()
    pred = clf.predict(emb)
    return "Normal speech 😊" if pred[0]==0 else "Possible speech issue ⚠️"

# ==========================
# GUI functions
# ==========================
def browse_file():
    file_path = filedialog.askopenfilename(
        title="Select audio file",
        filetypes=[("Audio files", "*.wav *.mp3 *.mp4 *.m4a")]
    )
    if file_path:
        entry_file.delete(0, tk.END)
        entry_file.insert(0, file_path)

def run_prediction():
    file_path = entry_file.get()
    if not file_path:
        messagebox.showwarning("Warning", "Please select an audio file!")
        return
    result = predict_audio(file_path)
    label_result.config(text=f"Prediction: {result}")

# ==========================
# GUI layout
# ==========================
root = tk.Tk()
root.title("🗣 Speech Disorder Detector")
root.geometry("500x250")
root.configure(bg="#f0f4f7")

# Header
header = tk.Label(root, text="Speech Disorder Detection", font=("Helvetica", 18, "bold"), bg="#f0f4f7", fg="#333")
header.pack(pady=10)

# File selection
frame_file = tk.Frame(root, bg="#f0f4f7")
frame_file.pack(pady=10)
entry_file = tk.Entry(frame_file, width=40, font=("Helvetica", 12))
entry_file.pack(side=tk.LEFT, padx=5)
btn_browse = tk.Button(frame_file, text="Browse", command=browse_file, bg="#4caf50", fg="white", font=("Helvetica", 10, "bold"))
btn_browse.pack(side=tk.LEFT, padx=5)

# Predict button
btn_predict = tk.Button(root, text="Predict", command=run_prediction, bg="#2196f3", fg="white", font=("Helvetica", 12, "bold"))
btn_predict.pack(pady=15)

# Result label
label_result = tk.Label(root, text="Prediction: ---", font=("Helvetica", 14), bg="#f0f4f7", fg="#e91e63")
label_result.pack(pady=10)

# Run app
root.mainloop()

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
